# Supercategory Clustering (Centroid-based, Agglomerative)

In [20]:
import os
import pandas as pd

parquet_path = "../data/processed/df_final.parquet"

if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path, engine="fastparquet")
else:
    train = pd.read_csv("../data/processed/train.csv")
    val = pd.read_csv("../data/processed/val.csv")
    test = pd.read_csv("../data/processed/test.csv")
    df = pd.concat([train, val, test], ignore_index=True)
    df.to_parquet(parquet_path, engine="fastparquet", index=False)

print("Rows:", len(df))
print("Unique labels:", df["label"].nunique())

Rows: 27550
Unique labels: 125


## Build label centroids from resume_text embeddings

In [21]:

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("./local_bert")

tmp = df[["label", "resume_text"]].copy()

MAX_PER_LABEL = 200

tmp = (
    tmp.groupby("label", group_keys=False)
       .apply(lambda g: g.sample(min(len(g), MAX_PER_LABEL), random_state=42))
       .reset_index(drop=True)
)

resume_embeddings = model.encode(
    tmp["resume_text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

tmp["idx"] = np.arange(len(tmp))
label_to_indices = tmp.groupby("label")["idx"].apply(list).to_dict()

centroids = []
labels_order = []

for lab, idxs in label_to_indices.items():
    centroids.append(resume_embeddings[idxs].mean(axis=0))
    labels_order.append(lab)

label_embeddings = np.vstack(centroids)
unique_labels = labels_order

print("Centroid embeddings shape:", label_embeddings.shape)


/var/folders/hh/cj64t6m14sd46vthykgl9lwc0000gn/T/ipykernel_57208/3598814152.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tmp.groupby("label", group_keys=False)
Batches: 100%|██████████| 480/480 [01:06<00:00,  7.19it/s]


Centroid embeddings shape: (125, 384)


In [22]:
# --- Cosine similarity between profession centroids ---

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sim_matrix = cosine_similarity(label_embeddings)

sim_df = pd.DataFrame(
    sim_matrix,
    index=unique_labels,
    columns=unique_labels
)

# For each label show 5 most similar professions
nearest = {}

for label in unique_labels:
    sims = sim_df.loc[label].sort_values(ascending=False)[1:6]
    nearest[label] = list(sims.index)

nearest_df = pd.DataFrame({
    "label": list(nearest.keys()),
    "nearest_labels": list(nearest.values())
})

nearest_df.head(15)

,label,nearest_labels
0,3d-моделлер,"[администратор проект, специалист it отдела, м..."
1,data scientist,"[qa engineer, программист с, программист c, те..."
2,frontend-разработчик,"[web-разработчик, веб-разработчик, веб-дизайне..."
3,html-верстальщик,"[веб-разработчик, веб-дизайнер, frontend-разра..."
4,it директор,"[руководитель отдела it, руководитель ит-отдел..."
5,it специалист,"[специалист ит, главный специалист, it-специал..."
6,it-инженер,"[инженер, it-специалист, системный инженер, ин..."
7,it-специалист,"[ит-специалист, специалист, технический специа..."
8,java-разработчик,"[программист java, тестировщик, html-верстальщ..."
9,php-программист,"[web-программист, веб-разработчик, web-разрабо..."


In [23]:
# --- Build graph-based supercategories from similarity ---

import networkx as nx
import numpy as np
import pandas as pd

SIM_THRESHOLD = 0.78  # можно будет регулировать

G = nx.Graph()

# Добавляем вершины
for label in unique_labels:
    G.add_node(label)

# Добавляем рёбра, если similarity выше порога
for i, label_i in enumerate(unique_labels):
    for j in range(i + 1, len(unique_labels)):
        label_j = unique_labels[j]
        sim = sim_df.iloc[i, j]
        if sim >= SIM_THRESHOLD:
            G.add_edge(label_i, label_j)

# Connected components = draft supercategories
components = list(nx.connected_components(G))

print("Number of components:", len(components))

component_sizes = [len(c) for c in components]
print("Component sizes:", sorted(component_sizes, reverse=True))

# Превращаем в DataFrame
draft_supercats = pd.DataFrame({
    "supercat_id": range(len(components)),
    "labels": [list(c) for c in components],
    "n_labels": [len(c) for c in components]
})

draft_supercats.sort_values("n_labels", ascending=False).head(10)

Number of components: 1
Component sizes: [125]


,supercat_id,labels,n_labels
0,0,"[it-инженер, программист 1c, ведущий программи...",125


In [27]:
# --- Mutual nearest neighbor grouping ---

TOP_K = 3

groups = []
visited = set()

for label in unique_labels:
    if label in visited:
        continue
        
    nearest = sim_df.loc[label].sort_values(ascending=False)[1:TOP_K+1].index.tolist()
    
    group = {label}
    
    for other in nearest:
        nearest_other = sim_df.loc[other].sort_values(ascending=False)[1:TOP_K+1].index.tolist()
        if label in nearest_other:
            group.add(other)
    
    visited.update(group)
    groups.append(group)

print("Number of micro-groups:", len(groups))
print("Group sizes:", sorted([len(g) for g in groups], reverse=True))

micro_df = pd.DataFrame({
    "group_id": range(len(groups)),
    "labels": [list(g) for g in groups],
    "size": [len(g) for g in groups]
})

micro_df.sort_values("size", ascending=False).head(10)

Number of micro-groups: 80
Group sizes: [4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


,group_id,labels,size
14,14,"[web-разработчик, веб-дизайнер, web-дизайнер, ...",4
36,36,"[менеджер, оператор, специалист, менеджер проект]",4
52,52,"[программист-разработчик, программист c, прогр...",4
7,7,"[ит-специалист, технический специалист, специа...",4
56,56,"[руководитель проекта, руководитель проект, ру...",4
58,58,"[руководитель отдела ит, руководитель отдела i...",4
28,28,"[инженер технической поддержк, сервисный инжен...",3
37,37,"[менеджер it-проект, менеджер проект, менеджер...",3
19,19,"[бизнес-аналитик, аналитик, системный аналитик]",3
72,72,"[тестировщик, специалист по защите информаци, ...",3


In [36]:
# --- Build functional supercategories from micro-groups ---

def classify_group(labels):
    text = " ".join(labels).lower()
    
    # Web
    if any(x in text for x in ["web", "frontend", "верст", "дизайн"]):
        return "web_development"
    
    # Software development
    if any(x in text for x in ["java", "python", "программист", "разработчик"]):
        return "software_development"
    
    # QA / Security
    if any(x in text for x in ["qa", "тест", "защите информации"]):
        return "qa_security"
    
    # Infrastructure
    if any(x in text for x in ["администратор", "linux", "системн", "devops", "сетев"]):
        return "infrastructure"
    
    # Industrial / field engineering
    if any(x in text for x in ["асу", "электрон", "электрик", "слаботоч", "связ", "монтаж"]):
        return "industrial_engineering"
    
    # Technical support
    if any(x in text for x in ["поддержк", "service", "сервис"]):
        return "tech_support"
    
    # Data / Analytics
    if any(x in text for x in ["data scientist", "аналитик"]):
        return "data_analytics"
    
    # Product / Project
    if any(x in text for x in ["product", "project", "менеджер проект", "интернет-проект", "координатор"]):
        return "product_project"
    
    # IT Governance
    if any(x in text for x in ["руководитель", "начальник"]):
        return "it_governance"
    
    # IT top management
    if any(x in text for x in ["директор"]):
        return "it_management"
    
    # Sales / Client
    if any(x in text for x in ["продаж", "клиент", "интернет-магазина"]):
        return "sales_client"
    
    # Marketing
    if any(x in text for x in ["seo", "маркетолог"]):
        return "digital_marketing"
    
    # Content / Design
    if any(x in text for x in ["3d", "контент", "писател"]):
        return "content_design"
    
    # Generic engineering
    if any(x in text for x in ["инженер", "ремонт"]):
        return "generic_engineering"
    
    # Generic IT
    if any(x in text for x in ["специалист it", "по it", "информационным технологиям"]):
        return "generic_it"
    
    return "other"


functional_groups = []

for _, row in micro_df.iterrows():
    category = classify_group(row["labels"])
    functional_groups.append((category, row["labels"]))

func_df = pd.DataFrame(functional_groups, columns=["supercategory", "labels"])

final_supercats = (
    func_df
    .groupby("supercategory")["labels"]
    .sum()
    .reset_index()
)

final_supercats["n_labels"] = final_supercats["labels"].apply(len)

final_supercats.sort_values("n_labels", ascending=False)

,supercategory,labels,n_labels
7,it_governance,"[начальник ит-отдела, главный специалист, руко...",23
6,infrastructure,"[модератор, администратор, техник, администрат...",22
13,software_development,"[программист java, java-разработчик, php-прогр...",18
10,product_project,"[product owner, product manager, project manag...",14
5,industrial_engineering,"[ведущий инженер, инженер слаботочных систем, ...",13
15,web_development,"[frontend-разработчик, веб-разработчик, web-ра...",9
12,sales_client,"[менеджер интернет-магазина, менеджер по сопро...",8
4,generic_it,"[ит-специалист, технический специалист, специа...",7
14,tech_support,"[инженер технической поддержк, сервисный инжен...",7
9,other,"[главный специалист, специалист ит, it специал...",6


In [37]:
# Inspect labels inside "other"

other_labels = final_supercats[
    final_supercats["supercategory"] == "other"
]["labels"].values[0]

print("Number of 'other' labels:", len(other_labels))
print(other_labels[:30])

Number of 'other' labels: 6
['главный специалист', 'специалист ит', 'it специалист', 'ведущий специалист', 'начинающий специалист', 'оператор видеонаблюдения']


In [39]:
# --- Final merge into 6 supercategories ---

final_merge_map = {
    "product_management": "product_management",
    "infra_support": "infra_support",
    "software_engineering": "software_engineering",
    "industrial_field": "industrial_field",
    "sales_marketing": "sales_marketing",
    "data_qa": "data_qa",
    "other_it": "infra_support",
    "content_creative": "sales_marketing"
}

merged_final = []

for _, row in final_merged.iterrows():
    old_cat = row["supercategory"]
    new_cat = final_merge_map.get(old_cat, old_cat)
    merged_final.append((new_cat, row["labels"]))

merged_final_df = pd.DataFrame(merged_final, columns=["supercategory", "labels"])

supercats_6 = (
    merged_final_df
    .groupby("supercategory")["labels"]
    .sum()
    .reset_index()
)

supercats_6["n_labels"] = supercats_6["labels"].apply(len)

supercats_6.sort_values("n_labels", ascending=False)

,supercategory,labels,n_labels
2,infra_support,"[ит-специалист, технический специалист, специа...",42
3,product_management,"[начальник ит-отдела, главный специалист, руко...",39
5,software_engineering,"[программист java, java-разработчик, php-прогр...",27
1,industrial_field,"[it-инженер, инженер, мастер по ремонту компью...",16
4,sales_marketing,"[3d-моделлер, контент-менеджер, технический пи...",14
0,data_qa,"[data scientist, аналитик бизнес-процесс, qa e...",8


In [40]:
# --- Build 10-12 supercategories (mid-granularity) ---

def refine_category(cat, labels):
    text = " ".join(labels).lower()

    # Split infra_support
    if cat == "infra_support":
        if any(x in text for x in ["devops", "linux", "сетев", "администратор", "системн"]):
            return "sysadmin_devops_network"
        if any(x in text for x in ["поддержк", "help", "service", "сервис", "оператор", "ремонт"]):
            return "tech_support_helpdesk"
        return "generic_it_ops"

    # Split product_management
    if cat == "product_management":
        if any(x in text for x in ["директор", "руководител", "начальник", "head"]):
            return "it_governance_leadership"
        return "project_product"

    # Split software_engineering
    if cat == "software_engineering":
        if any(x in text for x in ["frontend", "web", "верст", "дизайн"]):
            return "web_frontend"
        return "backend_general_dev"

    # Split industrial_field
    if cat == "industrial_field":
        if any(x in text for x in ["слаботоч", "связ", "монтаж", "электросвяз"]):
            return "telecom_low_voltage_install"
        return "hardware_industrial"

    # Split sales_marketing
    if cat == "sales_marketing":
        if any(x in text for x in ["seo", "маркет", "контент", "писател", "3d", "дизайн"]):
            return "marketing_content"
        return "sales_account"

    # Split data_qa (optional; we’ll check counts later)
    if cat == "data_qa":
        if any(x in text for x in ["qa", "тест", "защит"]):
            return "qa_security"
        return "data_analytics"

    return cat


rows = []
for _, row in supercats_6.iterrows():
    base_cat = row["supercategory"]
    for lab in row["labels"]:
        # refine based on single label (more precise than group text)
        new_cat = refine_category(base_cat, [lab])
        rows.append((lab, base_cat, new_cat))

mapping_mid = pd.DataFrame(rows, columns=["label", "base_supercat", "supercategory"])

mid_supercats = (
    mapping_mid
    .groupby("supercategory")["label"]
    .apply(list)
    .reset_index()
)

mid_supercats["n_labels"] = mid_supercats["label"].apply(len)
mid_supercats = mid_supercats.rename(columns={"label": "labels"})
mid_supercats.sort_values("n_labels", ascending=False)

,supercategory,labels,n_labels
4,it_governance_leadership,"[начальник ит-отдела, руководитель отдела, нач...",26
0,backend_general_dev,"[программист java, java-разработчик, php-прогр...",19
2,generic_it_ops,"[ит-специалист, технический специалист, специа...",18
9,sysadmin_devops_network,"[администратор, администратор баз данных, адми...",15
6,project_product,"[главный специалист, product owner, product ma...",13
3,hardware_industrial,"[it-инженер, инженер, мастер по ремонту компью...",10
10,tech_support_helpdesk,"[сервисный инженер, инженер технической поддер...",9
8,sales_account,"[менеджер интернет-магазина, менеджер по сопро...",8
12,web_frontend,"[frontend-разработчик, web-разработчик, веб-ди...",8
5,marketing_content,"[3d-моделлер, контент-менеджер, технический пи...",6


In [42]:
# --- Final stability merge for small categories ---

final_map = mapping_mid.copy()

merge_small = {
    "data_analytics": "technical_specialized",
    "qa_security": "technical_specialized",
    "marketing_content": "technical_specialized",
    "telecom_low_voltage_install": "technical_specialized",
    "hardware_industrial": "technical_specialized"
}

final_map["supercategory"] = final_map["supercategory"].replace(merge_small)

final_supercats_stable = (
    final_map
    .groupby("supercategory")["label"]
    .apply(list)
    .reset_index()
)

final_supercats_stable["n_labels"] = final_supercats_stable["label"].apply(len)

final_supercats_stable

,supercategory,label,n_labels
0,backend_general_dev,"[программист java, java-разработчик, php-прогр...",19
1,generic_it_ops,"[ит-специалист, технический специалист, специа...",18
2,it_governance_leadership,"[начальник ит-отдела, руководитель отдела, нач...",26
3,project_product,"[главный специалист, product owner, product ma...",13
4,sales_account,"[менеджер интернет-магазина, менеджер по сопро...",8
5,sysadmin_devops_network,"[администратор, администратор баз данных, адми...",15
6,tech_support_helpdesk,"[сервисный инженер, инженер технической поддер...",9
7,technical_specialized,"[data scientist, аналитик бизнес-процесс, qa e...",30
8,web_frontend,"[frontend-разработчик, web-разработчик, веб-ди...",8


In [43]:
df["supercategory"] = df["label"].map(
    dict(zip(final_map["label"], final_map["supercategory"]))
)

df["supercategory"].value_counts()

supercategory
sysadmin_devops_network     5514
generic_it_ops              4856
it_governance_leadership    3549
project_product             3342
technical_specialized       3303
backend_general_dev         2576
sales_account               1715
tech_support_helpdesk       1691
web_frontend                1004
Name: count, dtype: int64

In [46]:
final_mapping = dict(zip(final_map["label"], final_map["supercategory"]))

mapping_df = pd.DataFrame(
    final_mapping.items(),
    columns=["label", "supercategory"]
)

mapping_df.to_csv("../data/processed/label_to_supercategory_v1.csv", index=False)